# Native AHR — training and GNU benchmark controller

Purpose: execute the five authorized improvements with resumable competitive training, regular development matches, and a frozen 200-game final benchmark. Inputs: notebooks 00–03, fresh-kernel validation, pinned initial weights and prior training archives. Outputs: checkpoints, trajectories, development/final reports and verified local backup.

Expected runtime: approximately 3–5 days for the complete cycle, depending on move lengths and engine throughput. The first 512-game pilot is followed by adaptive continuation to at most 2048 cumulative games. A promising candidate can finish early after fresh confirmation. GPU 1 only; no automatic deployment.

Checkpoints: `checkpoints/pipeline-ahr-native-v4/` plus each child run. Active scientific state is saved every stage/game and within 250 optimizer steps/15 minutes. Latest two and best retained, one writer. Invalid manifests or changed sources cause an error.

Exact resume: preserve sources/configuration; start a fresh `ahr-go` kernel with `AHR_NATIVE_V4_RUN=1`, then Run All. Detached execution uses the same command in `launch.json`. Attach: `tmux attach -t ahr-go-native-v4`; detach: **Ctrl-b, then d**. Inspect the prior session and writer lock before resuming. Default Run All performs a short controller check and displays status.

Final games use untouched seeds, balanced colors, no handicap, 9×9 Chinese rules, komi 7.5 and one-second AHR thinking. UNKNOWN remains UNKNOWN. Report observed half wins separately from statistical support. Store final checkpoints and hash-verified local backup on the persistent VM; off-instance storage is deferred by the user.


In [ ]:
from pathlib import Path
import nbformat, os
PROJECT=Path.cwd().resolve()
while not (PROJECT/'notebooks/00_instance_setup_and_recovery.ipynb').is_file():
    if PROJECT.parent==PROJECT: raise RuntimeError('Run inside AHR Go workspace')
    PROJECT=PROJECT.parent
for cell in nbformat.read(PROJECT/'experiments/ahr_native_v4/00_shared.ipynb',4).cells:
    if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]): exec(compile(cell.source,'shared:ahr-export','exec'),globals())
v4_load_upstream(PROJECT,globals())
for name in ('01_native_ahr_and_tactics.ipynb','02_competitive_training.ipynb','03_gnugo_evaluation.ipynb'):
    v4_import_exports(PROJECT/V4_ROOT/name,globals())


## Immutable protocol and checkpointed controller

In [ ]:
from dataclasses import asdict
import datetime as dt, shutil, subprocess, shlex, tarfile

V4_CONTROLLER='pipeline-ahr-native-v4'
V4_TRAIN_RUN='ahr-native-v4-main'
V4_SESSION='ahr-go-native-v4'
V4_PROTOCOL_PATH='config/ahr_native_v4.json'
V4_PIPELINE_NOTEBOOK='experiments/ahr_native_v4/04_pipeline.ipynb'

def v4_now(): return dt.datetime.now(dt.timezone.utc).isoformat()

def v4_pipeline_read_protocol(project):
    protocol=v4_read_json(Path(project)/V4_PROTOCOL_PATH)
    if v4_digest({k:v for k,v in protocol.items() if k!='configuration_sha256'})!=protocol['configuration_sha256']: raise RuntimeError('Protocol digest mismatch')
    if protocol['sources']!=v4_source_identity(project): raise RuntimeError('Frozen source changed; inspect rather than restart')
    return protocol

def v4_pipeline_prepare(project):
    project=Path(project)
    if (project/V4_PROTOCOL_PATH).exists(): return v4_pipeline_read_protocol(project)
    validation=v4_read_json(v4_report_path(project,'delivery_validation.json'))
    if validation.get('status')!='PASS' or validation.get('sources')!=v4_source_identity(project): raise RuntimeError('Current fresh-kernel delivery validation required')
    for name in ('model','training_recovery','arena_recovery','pipeline_recovery','backup_recovery'):
        if validation.get(name,{}).get('status')!='PASS': raise RuntimeError('Missing validated component '+name)
    config=asdict(V4TrainConfig())
    if config.get('scheduled_games')!=2048 or config.get('device')!='cuda:1': raise RuntimeError('Expected 2048 maximum games on GPU1')
    if torch.cuda.device_count()<2: raise RuntimeError('GPU1 unavailable')
    if shutil.disk_usage(project).free<15*1024**3: raise RuntimeError('Insufficient persistent storage')
    protocol={'schema':V4_SCHEMA,'controller':V4_CONTROLLER,'training_run':V4_TRAIN_RUN,'training':config,
        'training_targets':[128,256,512,1024,2048],'development_games':32,'confirmation_games':64,
        'confirmation_gate':0.625,'final_games':200,'final_target':0.5,'evaluation_device':'cuda:1',
        'rules':RULESET,'size':9,'komi':7.5,'handicap':0,'ahr_thinking_seconds':1.0,'gnu_level':1,
        'selection':'Greatest development wins/planned, margin tie break. Fresh confirmation of promising candidates. Freeze before final lockbox.',
        'unknown_policy':'UNKNOWN retained; no invented win/loss; wins/planned conservative.',
        'scope':'Native AHR Queen/Worker/Scout; GNU opposition only; no JEPA predictor or GNU policy targets.',
        'off_instance_backup':'DEFERRED_BY_USER_PENDING_HUGGING_FACE_DESTINATION',
        'sources':v4_source_identity(project),'validation_sha256':file_sha256(v4_report_path(project,'delivery_validation.json')),'created_at_utc':v4_now()}
    protocol['configuration_sha256']=v4_digest(protocol);v4_atomic_json(project/V4_PROTOCOL_PATH,protocol)
    return protocol

def v4_pipeline_load_unlocked(project,protocol):
    directory=Path(project)/'checkpoints'/V4_CONTROLLER;path=directory/'manifest.json'
    if not path.exists():
        if list(directory.glob('*.pt')): raise RuntimeError('Controller manifest missing')
        return None
    manifest=v4_read_json(path)
    entry=next((e for e in manifest.get('entries',[]) if e['file']==manifest.get('latest')),None)
    if entry is None: raise RuntimeError('Latest controller checkpoint not committed')
    state=_checkpoint_entry_state(directory,entry)
    if state is None: raise RuntimeError('Invalid latest controller checkpoint; inspect rather than restart')
    if state.get('config')!=protocol: raise RuntimeError('Controller identity differs')
    if state['sampler']['phase'] not in ('training','development','confirmation','final','bundle','complete'): raise RuntimeError('Invalid phase')
    if not 0<=state['sampler']['target_index']<len(protocol['training_targets']): raise RuntimeError('Invalid stage position')
    restore_rng(state['rng']);return state

def v4_pipeline_status(project):
    path=v4_report_path(project,'pipeline_status.json')
    return v4_read_json(path) if path.exists() else {'status':'NOT_STARTED','session':V4_SESSION}

def v4_pipeline_commit(project,state,best=False):
    state['global_step']+=1;state['rng']=capture_rng()
    entry=_save_checkpoint_unlocked(Path(project)/'checkpoints'/V4_CONTROLLER,state,best=best,
        resume_cell='04_pipeline: unchanged source/config, fresh ahr-go kernel, AHR_NATIVE_V4_RUN=1, Run All')
    report={'schema':V4_SCHEMA,'status':state['metrics'].get('status','RUNNING'),'phase':state['sampler']['phase'],
        'target_games':state['config']['training_targets'][state['sampler']['target_index']],
        'training_run':V4_TRAIN_RUN,'global_step':state['global_step'],'best_development':state['metrics'].get('best_development'),
        'development':state['metrics']['development'],'final':state['metrics'].get('final'),'last_error':state['metrics'].get('last_error'),
        'checkpoint':entry,'updated_at_utc':v4_now(),'session':V4_SESSION}
    v4_atomic_json(v4_report_path(project,'pipeline_status.json'),report)
    print(json.dumps({k:report[k] for k in ('status','phase','target_games','global_step','updated_at_utc')}),flush=True)
    return entry

def v4_development_rank(receipt):
    margin=receipt.get('mean_score_margin')
    return (float(receipt.get('wins_over_planned',0)),float(margin) if margin is not None else -999.0)

def v4_pipeline_local_bundle(project,final):
    project=Path(project)
    gnugo10_binary_identity(project)
    roots=[project/'notebooks',project/V4_ROOT,project/'reports/ahr_native_v4',project/GNUGO10_BINARY_RELATIVE]
    roots += [project/'checkpoints'/V4_CONTROLLER, project/'checkpoints'/V4_TRAIN_RUN]
    for directory in (project/'checkpoints').iterdir():
        if directory.is_dir() and directory.name.startswith(('ahr-native-v4-dev-','ahr-native-v4-confirm-','ahr-native-v4-final-')): roots.append(directory)
    for run in ('ahr-v3-B-s01','ahr-v3-B-s02','ahr-v3-B-s03'):
        path=project/'reports/ahr_jepa_v3'/(run+'_training.json')
        if not path.is_file(): raise RuntimeError('Missing prior training provenance '+run)
        roots.append(path)
    for run in ('go09-first-selfplay-v1','ahr-v3-B-s01','ahr-v3-B-s02','ahr-v3-B-s03'):
        directory=project/'checkpoints'/run
        if directory.exists(): roots.append(directory)
        directory=project/'experiments/ahr_jepa_v3/data/trajectories'/run
        if directory.exists(): roots.append(directory)
    for relative in (V4_PROTOCOL_PATH,'config/ahr_native_v4_final_lockbox.json','config/project.json','config/ahr_jepa_v3.json','experiments/ahr_jepa_v3/data/curriculum.json','experiments/ahr_jepa_v3/data/curriculum.manifest.json'):
        path=project/relative
        if path.exists(): roots.append(path)
    if not (project/'config/ahr_native_v4_final_lockbox.json').is_file(): raise RuntimeError('Final lockbox declaration missing')
    files=sorted({p for root in roots for p in ([root] if root.is_file() else root.rglob('*')) if p.is_file() and p.name!='.checkpoint.lock' and '.ipynb_checkpoints' not in p.parts})
    hashes={str(p.relative_to(project)):file_sha256(p) for p in files}
    name='ahr-native-v4-final-'+v4_digest({'files':hashes,'model':final['checkpoint_sha256']})[:12]
    target=project/'exports/bundles'/(name+'.tar.gz');target.parent.mkdir(parents=True,exist_ok=True)
    temporary=target.with_name('.'+target.name+'.tmp')
    try:
        with tarfile.open(temporary,'w:gz') as archive:
            for path in files: archive.add(path,arcname=str(path.relative_to(project)),recursive=False)
        os.replace(temporary,target)
    finally:
        if temporary.exists(): temporary.unlink()
    restored=project/'exports/restored'/name;restored.mkdir(parents=True,exist_ok=True)
    with tarfile.open(target,'r:gz') as archive: archive.extractall(restored,filter='data')
    for relative,expected in hashes.items():
        if file_sha256(restored/relative)!=expected: raise RuntimeError('Restore hash mismatch '+relative)
    namespace={'__name__':'restored_native_ahr'}
    v4_import_exports(restored/V4_ROOT/'00_shared.ipynb',namespace);namespace['v4_load_upstream'](restored,namespace)
    for notebook in ('01_native_ahr_and_tactics.ipynb','02_competitive_training.ipynb'):
        namespace['v4_import_exports'](restored/V4_ROOT/notebook,namespace)
    checkpoint=Path(final['checkpoint_path']);relative=checkpoint.relative_to(project) if checkpoint.is_absolute() else checkpoint
    model,metadata=namespace['v4_load_model'](restored,V4_TRAIN_RUN,device='cpu',checkpoint_path=restored/relative)
    with torch.inference_mode(): output=model(namespace['encode_go_batch']([namespace['initial_state'](9,7.5)],'cpu'))
    if not bool(torch.isfinite(output['value']).all()): raise RuntimeError('Restored CPU inference failed')
    training_state,training_entry=namespace['v4_read_training_checkpoint'](restored/'checkpoints'/V4_TRAIN_RUN)
    if training_state is None: raise RuntimeError('Restored main training checkpoint missing')
    namespace['v4_verify_replay_references'](restored,training_state['replay_entries'])
    declared_config=namespace['V4TrainConfig'](**training_state['config'])
    curriculum=namespace['v4_tactical_curriculum'](restored)
    prior_entries,prior_metadata=namespace['v4_training_prior'](restored,declared_config)
    identity=namespace['v4_training_identity'](restored,declared_config,curriculum,prior_metadata)
    if identity!=training_state['identity']: raise RuntimeError('Restored training input identity mismatch')
    cpu_values=copy.deepcopy(training_state['config']);cpu_values['device']='cpu'
    cpu_components,_=namespace['v4_training_components'](restored,namespace['V4TrainConfig'](**cpu_values),training_state['model_config'])
    namespace['v4_restore_training'](training_state,cpu_components)
    for expected,component in zip(('model','optimizer','scheduler','scaler'),cpu_components):
        if not nested_equal(arena_cpu(component.state_dict()),arena_cpu(training_state[expected])):
            raise RuntimeError('Restored scientific state mismatch '+expected)
    restored_controller=v4_pipeline_load_unlocked(restored,v4_read_json(restored/V4_PROTOCOL_PATH))
    if restored_controller is None: raise RuntimeError('Restored controller state missing')
    if v4_read_json(restored/'config/ahr_native_v4_final_lockbox.json')!=v4_read_json(project/'config/ahr_native_v4_final_lockbox.json'):
        raise RuntimeError('Restored final lockbox mismatch')
    report={'status':'VERIFIED_LOCAL','bundle':str(target.relative_to(project)),'sha256':file_sha256(target),'bytes':target.stat().st_size,
        'files':len(files),'restored_workspace':str(restored.relative_to(project)),'cpu_inference':'PASS',
        'full_scientific_state_restore':'PASS','prior_input_games':len(prior_entries),'replay_references_verified':len(training_state['replay_entries']),
        'controller_restored_step':restored_controller['global_step'],'final_lockbox_preserved':True,
        'runtime_dependencies':'Install matching Python/.venv packages and the pinned GNU Go binary before restored training; identities recorded in validation/protocol.',
        'off_instance_backup':'DEFERRED_BY_USER_PENDING_HUGGING_FACE_DESTINATION'}
    v4_atomic_json(v4_report_path(project,'local_bundle_final.json'),report);return report

def v4_run_pipeline(project):
    project=Path(project);protocol=v4_pipeline_read_protocol(project)
    with checkpoint_lock(project/'checkpoints'/V4_CONTROLLER):
        state=v4_pipeline_load_unlocked(project,protocol)
        if state is None:
            state={'model':{},'optimizer':{},'scheduler':{},'scaler':{},'rng':capture_rng(),'global_step':0,'config':protocol,
                'sampler':{'phase':'training','target_index':0},'metrics':{'status':'RUNNING','development':[],'best_development':None,'last_error':None},
                'child_state_contract':'Active model/optimizer/RNG/replay/game state is checkpointed by each child; controller checkpoints orchestration boundaries.'}
            v4_pipeline_commit(project,state,best=True)
        try:
            while state['sampler']['phase']!='complete':
                phase=state['sampler']['phase'];index=state['sampler']['target_index'];target=protocol['training_targets'][index]
                state['metrics']['status']='RUNNING';state['metrics']['last_error']=None
                if phase=='training':
                    state['metrics']['training']=v4_run_training(project,V4_TRAIN_RUN,V4TrainConfig(**protocol['training']),run_until_games=target)
                    state['sampler']['phase']='development'
                elif phase=='development':
                    receipt=v4_run_gnu_evaluation(project,V4_TRAIN_RUN,f'ahr-native-v4-dev-g{target:04d}',split='development',games=protocol['development_games'],device=protocol['evaluation_device'])
                    state['metrics']['development'].append(receipt);best=state['metrics']['best_development']
                    if best is None or v4_development_rank(receipt)>v4_development_rank(best): state['metrics']['best_development']=receipt
                    if receipt['wins_over_planned']>=protocol['confirmation_gate']:
                        state['metrics']['confirmation_candidate']=receipt;state['sampler']['phase']='confirmation'
                    elif index+1<len(protocol['training_targets']): state['sampler'].update(phase='training',target_index=index+1)
                    else: state['sampler']['phase']='final'
                elif phase=='confirmation':
                    candidate=state['metrics']['confirmation_candidate']
                    receipt=v4_run_gnu_evaluation(project,V4_TRAIN_RUN,f'ahr-native-v4-confirm-g{target:04d}',split='development',games=protocol['confirmation_games'],checkpoint_path=candidate['checkpoint_path'],device=protocol['evaluation_device'])
                    state['metrics']['development'].append(receipt)
                    if receipt['wins_over_planned']>=0.6: state['metrics']['best_development']=receipt;state['sampler']['phase']='final'
                    elif index+1<len(protocol['training_targets']): state['sampler'].update(phase='training',target_index=index+1)
                    else: state['sampler']['phase']='final'
                elif phase=='final':
                    best=state['metrics']['best_development']
                    if best is None: raise RuntimeError('No development-selected model')
                    state['metrics']['final']=v4_run_gnu_evaluation(project,V4_TRAIN_RUN,'ahr-native-v4-final-200',split='final',games=protocol['final_games'],checkpoint_path=best['checkpoint_path'],device=protocol['evaluation_device'])
                    state['sampler']['phase']='bundle'
                elif phase=='bundle':
                    final=state['metrics']['final'];state['metrics']['local_bundle']=v4_pipeline_local_bundle(project,final)
                    state['metrics']['status']='TARGET_OBSERVED' if final['observed_half_wins'] else 'CYCLE_COMPLETE_TARGET_NOT_REACHED'
                    state['sampler']['phase']='complete'
                    completion={'schema':V4_SCHEMA,'status':state['metrics']['status'],'all_five_steps_executed':True,
                        'observed_half_wins':final['observed_half_wins'],'statistically_supported_above_half':final['statistically_supported_above_half'],
                        'final':final,'development':state['metrics']['development'],'local_bundle':state['metrics']['local_bundle'],
                        'completed_at_utc':v4_now(),'further_work_required':not final['observed_half_wins']}
                    v4_atomic_json(v4_report_path(project,'completion.json'),completion)
                else: raise RuntimeError('Invalid phase '+phase)
                v4_pipeline_commit(project,state,best=state['sampler']['phase']=='complete')
        except BaseException as error:
            state['metrics']['status']='INTERRUPTED_OR_FAILED';state['metrics']['last_error']=type(error).__name__+': '+str(error)
            v4_pipeline_commit(project,state);raise
    return v4_pipeline_status(project)

def v4_validate_pipeline_recovery(project):
    project=Path(project);run='ahr-native-v4-controller-smoke-'+uuid.uuid4().hex[:10];directory=project/'checkpoints'/run
    state={'model':{},'optimizer':{},'scheduler':{},'scaler':{},'rng':capture_rng(),'global_step':1,'config':{'smoke':True},
        'sampler':{'phase':'training','target_index':0},'metrics':{'child_resume_step':41}}
    with checkpoint_lock(directory):
        first=_save_checkpoint_unlocked(directory,state,best=True,resume_cell='04 controller validation')
        state['global_step']=2;state['metrics']['child_resume_step']=42
        second=_save_checkpoint_unlocked(directory,state,resume_cell='04 controller validation')
        resumed=_checkpoint_entry_state(directory,second)
        if resumed is None or resumed['metrics']['child_resume_step']!=42: raise AssertionError('Controller resume mismatch')
        declaration=v4_read_json(directory/'manifest.json')
        if len(declaration['entries'])!=2 or declaration['best']!=first['file']: raise AssertionError('Retention mismatch')
        payload=(directory/second['file']).read_bytes();atomic_write_bytes(directory/second['file'],b'invalid')
        if _checkpoint_entry_state(directory,second) is not None: raise AssertionError('Corruption accepted')
        atomic_write_bytes(directory/second['file'],payload)
    report={'status':'PASS','run_id':run,'resumed_step':2,'resumed_child_step':42,'latest_two_and_best':True,'corruption_rejected':True}
    v4_atomic_json(v4_report_path(project,'pipeline_recovery.json'),report);return report

def v4_launch_detached(project):
    project=Path(project);protocol=v4_pipeline_read_protocol(project)
    with checkpoint_lock(project/'checkpoints'/(V4_CONTROLLER+'-launch')):
        if subprocess.run(['tmux','has-session','-t',V4_SESSION],capture_output=True).returncode==0: raise RuntimeError('Existing tmux session; inspect before launching another writer')
        command=['env','AHR_NATIVE_V4_RUN=1','CUBLAS_WORKSPACE_CONFIG=:4096:8',sys.executable,'-u','-m','jupyter','nbconvert','--execute','--to','notebook','--inplace','--ExecutePreprocessor.kernel_name=ahr-go','--ExecutePreprocessor.timeout=-1',str(project/V4_PIPELINE_NOTEBOOK)]
        log=v4_report_path(project,'controller.log')
        subprocess.run(['tmux','new-session','-d','-s',V4_SESSION,'-n','pipeline','-c',str(project),'exec '+shlex.join(command)+' >> '+shlex.quote(str(log))+' 2>&1'],check=True)
        subprocess.run(['tmux','set-window-option','-t',V4_SESSION+':pipeline','remain-on-exit','on'],check=True)
        receipt={'status':'LAUNCHED','session':V4_SESSION,'configuration_sha256':protocol['configuration_sha256'],
            'attach':'tmux attach -t '+V4_SESSION,'detach':'Ctrl-b, then d','controller_log':str(log.relative_to(project)),
            'resume':'Unchanged source/config; fresh ahr-go kernel with AHR_NATIVE_V4_RUN=1; Run All in 04_pipeline.ipynb',
            'device':'cuda:1','training_targets':protocol['training_targets'],'final_games':protocol['final_games'],'launched_at_utc':v4_now()}
        v4_atomic_json(v4_report_path(project,'launch.json'),receipt);return receipt


## Validate or resume

In [ ]:
if os.environ.get('AHR_NATIVE_V4_RUN')=='1':
    result=v4_run_pipeline(PROJECT)
else:
    result={'controller_recovery':v4_validate_pipeline_recovery(PROJECT),'pipeline':v4_pipeline_status(PROJECT)}
print(json.dumps(result,indent=2,default=str))
